# 👁️ Azure AI Vision — Lab AI-102

**Objectif**: Analyser des images et extraire du texte avec Azure AI Vision.

## Compétences AI-102 couvertes
- Analyser des images (légendes, objets, personnes, tags)
- OCR: extraire du texte imprimé et manuscrit
- Comprendre les scores de confiance
- Légendes denses et smart crops
- Traiter des images depuis bytes ou URLs

In [ ]:
%pip install azure-ai-vision-imageanalysis python-dotenv Pillow requests -q

In [ ]:
import os
import requests
from dotenv import load_dotenv
from azure.ai.vision.imageanalysis import ImageAnalysisClient
from azure.ai.vision.imageanalysis.models import VisualFeatures
from azure.core.credentials import AzureKeyCredential
from PIL import Image
import io

load_dotenv('../.env')

client = ImageAnalysisClient(
    endpoint=os.getenv('AZURE_VISION_ENDPOINT'),
    credential=AzureKeyCredential(os.getenv('AZURE_VISION_KEY'))
)

# Image de test publique
SAMPLE_IMAGE_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/1200px-Cat03.jpg"

print('✅ Client Azure AI Vision initialisé')

## 1. Analyse complète d'image depuis URL

In [ ]:
# AI-102: VisualFeatures détermine ce que l'API analyse
# Choisir uniquement les features nécessaires pour optimiser les coûts

result = client.analyze_from_url(
    image_url=SAMPLE_IMAGE_URL,
    visual_features=[
        VisualFeatures.CAPTION,
        VisualFeatures.TAGS,
        VisualFeatures.OBJECTS,
        VisualFeatures.PEOPLE,
        VisualFeatures.DENSE_CAPTIONS,
    ],
    gender_neutral_caption=True
)

# Légende principale
if result.caption:
    print(f"Légende: '{result.caption.text}'")
    print(f"Confiance: {result.caption.confidence:.2%}")

# Tags
if result.tags:
    print(f"\nTags ({len(result.tags.list)}):")
    for tag in sorted(result.tags.list, key=lambda t: t.confidence, reverse=True)[:8]:
        print(f"  • {tag.name}: {tag.confidence:.0%}")

# Objets détectés
if result.objects:
    print(f"\nObjets ({len(result.objects.list)}):")
    for obj in result.objects.list:
        name = obj.tags[0].name if obj.tags else 'unknown'
        confidence = obj.tags[0].confidence if obj.tags else 0
        print(f"  • {name}: {confidence:.0%}")

# Personnes
if result.people:
    print(f"\nPersonnes détectées: {len(result.people.list)}")

## 2. OCR — Extraction de texte (Read API)

In [ ]:
# AI-102: VisualFeatures.READ extrait le texte de l'image
# Supporte: texte imprimé, manuscrit, multi-langues
# Retourne: pages > blocs > lignes > mots avec positions

# Utiliser une image avec du texte
text_image_url = "https://learn.microsoft.com/azure/ai-services/computer-vision/media/quickstarts/presentation.png"

result = client.analyze_from_url(
    image_url=text_image_url,
    visual_features=[VisualFeatures.READ]
)

if result.read:
    print("Texte extrait:")
    print("-" * 40)
    for block in result.read.blocks:
        for line in block.lines:
            print(f"  {line.text}")
            # Afficher les mots avec scores de confiance
            for word in line.words[:3]:  # Limiter l'affichage
                print(f"    '{word.text}' (confiance: {word.confidence:.0%})")
else:
    print("Aucun texte détecté")

## 3. Analyse depuis bytes (image locale)

In [ ]:
# AI-102: Analyser une image depuis des bytes (fichier local)

# Télécharger l'image pour simuler un fichier local
response = requests.get(SAMPLE_IMAGE_URL, timeout=10)
image_bytes = response.content

result = client.analyze(
    image_data=image_bytes,
    visual_features=[VisualFeatures.CAPTION, VisualFeatures.TAGS],
    gender_neutral_caption=True
)

print(f"Analyse depuis bytes: {len(image_bytes)} octets")
if result.caption:
    print(f"Légende: {result.caption.text}")
if result.tags:
    print(f"Tags: {[t.name for t in result.tags.list[:5]]}")

## 4. Légendes denses (Dense Captions)

In [ ]:
# AI-102: Dense Captions génère des légendes pour chaque région de l'image
# Utile pour l'accessibilité et la description détaillée

result = client.analyze_from_url(
    image_url=SAMPLE_IMAGE_URL,
    visual_features=[VisualFeatures.DENSE_CAPTIONS]
)

if result.dense_captions:
    print(f"Légendes denses ({len(result.dense_captions.list)}):")
    for i, caption in enumerate(result.dense_captions.list):
        bb = caption.bounding_box
        region = f"[x:{bb.x}, y:{bb.y}, w:{bb.width}, h:{bb.height}]" if bb else ""
        print(f"  {i+1}. '{caption.text}' ({caption.confidence:.0%}) {region}")

## 5. Smart Crops — Recadrage intelligent

In [ ]:
# AI-102: Smart Crops identifie la zone d'intérêt pour un ratio cible
# Utile pour générer des thumbnails adaptés à différents formats (carré, paysage, portrait)

result = client.analyze_from_url(
    image_url=SAMPLE_IMAGE_URL,
    visual_features=[VisualFeatures.SMART_CROPS],
    smart_crops_aspect_ratios=[1.0, 1.78, 0.75]  # Carré, 16:9, Portrait
)

if result.smart_crops:
    print("Zones de recadrage suggérées:")
    for crop in result.smart_crops.list:
        bb = crop.bounding_box
        print(f"  Ratio {crop.aspect_ratio}: x={bb.x}, y={bb.y}, w={bb.width}, h={bb.height}")

## Résumé AI-102 — Azure AI Vision

| VisualFeature | Description | Usage documentaire |
|--------------|-------------|-------------------|
| `CAPTION` | Légende globale de l'image | Description automatique |
| `DENSE_CAPTIONS` | Légendes par région | Accessibilité |
| `READ` | OCR texte imprimé/manuscrit | Numérisation documents |
| `TAGS` | Tags/étiquettes | Classification |
| `OBJECTS` | Objets avec bounding boxes | Inventaire |
| `PEOPLE` | Détection de personnes | Sécurité/RH |
| `SMART_CROPS` | Recadrage intelligent | Thumbnails |